In [ ]:
import torch
from torch.nn.functional import logsigmoid
import numpy as np
from scipy.stats import norm
from tqdm.autonotebook import tqdm
import os, csv

# set a device + specify number of seeds, uniform-mixing & CI target coverage + spread of budgets
device = "cuda" if torch.cuda.is_available() else "cpu"
N_SEEDS, ALPHA = 1000, 0.05
BUDGET_PROPS = np.round(np.linspace(0.0, 0.25, 11)[1:], decimals=3)

# which dataset are we running?
dataset = "bodmas"

# filename for this simulation
fname = f"{dataset}_baselines.csv"

# dataframe to store all of our results
columns = ["dataset", "budget", "tau", "method", "seed", "thetahat", "coverage", "width", "n_labels"]
if fname not in os.listdir("../logs"):
    with open(f"../logs/{fname}", "wt") as file:
        file.write(",".join(columns) + "\n")
        
# dummy tensors for brevity
ZERO = torch.tensor(0.0, device=device)

In [ ]:
#### LOAD IN OUR DATA

# load in X_test and y_test + initial weights for OLR
Xs = torch.load(f"../datasets/{dataset}/X_test.pt").to(device) # (N, D)
ys = torch.load(f"../datasets/{dataset}/y_test.pt").to(device) # (N,)
beta_init = torch.load(f"../datasets/{dataset}/beta_init.pt").to(device) # (D,)

# get our finite-population size
N, D = Xs.shape

In [ ]:
#### BASELINES - ONLINE PPI + UNIFORM

# iterate through our budgets
for budget_idx, budget in enumerate(BUDGET_PROPS):
    
    # how many labels can we collect?
    n = int(budget * N)
    
    #### initialize data structures
    
    # set a seed for reproducibility
    torch.random.manual_seed(858 + budget_idx)

    # how many labels have we used up already
    obs_counts = torch.zeros(size=(N_SEEDS, 1), dtype=torch.float32, device=device)

    # recurring sum (WITHOUT NORMALIZATION) of thetahat (need div by N) and sigmahat^2 (need div by N^2)
    thetahats_unnormalized_unif = torch.zeros(
        size=(N_SEEDS, 1), device=device, dtype=torch.float32) # (N_SEEDS, 1)
    sigmahats_sq_unnormalized_unif = torch.zeros(
        size=(N_SEEDS, 1), device=device, dtype=torch.float32) # (N_SEEDS, 1)
    thetahats_unnormalized_ppi = torch.zeros(
        size=(N_SEEDS, 1), device=device, dtype=torch.float32) # (N_SEEDS, 1)
    sigmahats_sq_unnormalized_ppi = torch.zeros(
        size=(N_SEEDS, 1), device=device, dtype=torch.float32) # (N_SEEDS, 1)
    
    # initialized LR coefficients + let's use identity initial diagonal precisions
    betas = torch.tile(beta_init, (N_SEEDS, 1)) # (N)
    diag_precisions = torch.full(
        size=(N_SEEDS, D), fill_value=1.0, device=device, dtype=torch.float32) # (N_SEEDS, D)
    
    #### sequential active inference-type (need loop thru full dataset, yes budget stabilization if "sai")
    for t in tqdm(range(N), desc=f"Budget {budget_idx + 1} of {len(BUDGET_PROPS)}"):
        
        #### PART 1. computing q_t(j) probabilities + sampling probabilities
        
        # under uniform/ppi sampling, pi_t = n / N always.
        pi_t = n / N

        # are we going to query this data point or not?
        xi_t = torch.bernoulli(
            torch.full(size=(N_SEEDS, 1), fill_value=pi_t, device=device)) # (N_SEEDS, 1)
        
        #### PART 2. UPDATING SEQUENTIAL ACTIVE INFERENCE-TYPE ESTIMATOR + BOOKKEEPING
        
        # a. compute current ML predictions
        logits = betas @ Xs.T # (N_SEEDS, N)
        fxs = torch.sigmoid(logits) # (N_SEEDS, N)

        # b. update our means and variances for unif + ppi estimators
        thetahats_unnormalized_unif += ((xi_t / pi_t) * ys[t])
        sigmahats_sq_unnormalized_unif += (xi_t * (1.0 - pi_t) / (pi_t ** 2) ) * (ys[t] ** 2)
        thetahats_unnormalized_ppi += ((xi_t / pi_t) * (ys[t] - fxs[:,t:t+1])) + fxs[:,t:t+1]
        sigmahats_sq_unnormalized_ppi += (xi_t * (1.0 - pi_t) / (pi_t ** 2) ) * ((ys[t] - fxs[:,t:t+1]) ** 2)

        # c. did we exhaust budget? also, did we hit a big-residual datapoint?
        obs_counts += xi_t
        
        #### PART 3: updating our online logistic regression
        
        # a. convert xi_t to bool
        xi_t = xi_t.to(bool).squeeze() # (N_NEW,)
        
        # b. fhat * (1 - fhat); note that diag(xx^T) = (x_1^2, \dots, x_D^2)
        w_Is = fxs[:,t:t+1][xi_t] * (1.0 - fxs[:,t:t+1][xi_t]) # (N_FLIP_HEADS, 1)
        x_t = Xs[t] # (D,)
        diag_precisions[xi_t] += w_Is * (x_t ** 2) # (N_SEEDS, D)
        betas[xi_t] += (ys[t] - fxs[:,t:t+1][xi_t]) * x_t / diag_precisions[xi_t] # (N_SEEDS, D)
        
    #### COMPUTING UNIF + PPI ESTIMATOR SUMMARIES
    
    # dummy tensor for brevity
    ZERO = torch.tensor(0.0, device=device)
    
    #### 1. UNIFORM ESTIMATOR
    
    # a. get our mean estimate + variance estimate
    thetahats_unif = thetahats_unnormalized_unif / N
    sigmahats_unif = torch.sqrt(torch.maximum(sigmahats_sq_unnormalized_unif, ZERO) / N)
    sigmahats_bar_unif = sigmahats_unif / torch.sqrt(torch.tensor(N, device=device))

    # b. get our z-score + the lower/upper limits of our interval
    z_score = norm.ppf(1 - (ALPHA / 2))
    lbs = thetahats_unif - z_score * sigmahats_bar_unif
    ubs = thetahats_unif + z_score * sigmahats_bar_unif
    widths = (ubs - lbs).cpu().numpy()
    coverages = ((lbs <= ys.mean(dim=-1, keepdim=True)) & (ys.mean(dim=-1, keepdim=True) <= ubs)).cpu().numpy()
    
    # c. for file writing purposes, convert thetahats + obs_counts to numpy
    thetahats_unif = thetahats_unif.cpu().numpy()
    obs_counts = obs_counts.cpu().numpy()
    
    # d. format results + directly write to file
    results = [
        [dataset, budget, 0.0, "uniform", seed, 
         thetahats_unif[seed][0], coverages[seed][0], widths[seed][0], obs_counts[seed][0]
        ] for seed in range(N_SEEDS)]
    with open(f"../logs/{fname}", "a", newline="") as file:
        writer = csv.writer(file)
        writer.writerows(results)
        
    #### 2. PPI ESTIMATOR
    
    # a. get our mean estimate + variance estimate
    thetahats_ppi = thetahats_unnormalized_ppi / N
    sigmahats_ppi = torch.sqrt(torch.maximum(sigmahats_sq_unnormalized_ppi, ZERO) / N)
    sigmahats_bar_ppi = sigmahats_ppi / torch.sqrt(torch.tensor(N, device=device))

    # b. get our z-score + the lower/upper limits of our interval
    z_score = norm.ppf(1 - (ALPHA / 2))
    lbs = thetahats_ppi - z_score * sigmahats_bar_ppi
    ubs = thetahats_ppi + z_score * sigmahats_bar_ppi
    widths = (ubs - lbs).cpu().numpy()
    coverages = ((lbs <= ys.mean(dim=-1, keepdim=True)) & (ys.mean(dim=-1, keepdim=True) <= ubs)).cpu().numpy()
    
    # c. for file writing purposes, convert thetahats ppi also to numpy.
    thetahats_ppi = thetahats_ppi.cpu().numpy()
    
    # d. format results + directly write to file
    results = [
        [dataset, budget, 0.0, "ppi", seed, 
         thetahats_ppi[seed][0], coverages[seed][0], widths[seed][0], obs_counts[seed][0]
        ] for seed in range(N_SEEDS)]
    with open(f"../logs/{fname}", "a", newline="") as file:
        writer = csv.writer(file)
        writer.writerows(results)